# 🌐 Язык / Language

**RU.** Выберите язык в ячейке ниже: `LANG = "ru"` или `LANG = "en"`, затем запустите блокнот
целиком (Run All). На выбранном языке будет весь текст: пояснения, выводы ячеек, подписи графиков.

**EN.** Choose the language in the cell below: `LANG = "ru"` or `LANG = "en"`, then run the whole
notebook (Run All). Everything will be in that language: the explanations, the cell outputs and the
chart labels.


In [ ]:
LANG = "ru"   # "ru" -- русский | "en" -- English

%load_ext autoreload
%autoreload 2

import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seasonal_toolbox as sx
sx.set_lang(LANG)
from seasonal_toolbox import *

# Путь к бинарнику X-13, если он у вас есть. Без него работают встроенные реализации.
# Path to the X-13 binary if you have one. Without it the built-in implementations are used.
sx.X13_BIN = os.getenv("PATH_X13", "")

pal = econ_style("light")

TXT = {
    "hello": {"ru": "seasonal_toolbox {0} | x13as: {1}",
              "en": "seasonal_toolbox {0} | x13as: {1}"},
    "no_x13": {"ru": "не найден, используются встроенные реализации",
               "en": "not found, the built-in implementations are used"},
}
print(tx(TXT["hello"]).format(sx.__version__, sx.X13_BIN or tx(TXT["no_x13"])))


In [ ]:
MD = {
"ru": r"""
# Сезонная корректировка показателя: рабочий шаблон

Один показатель — один блокнот. Копируется на каждый новый ряд, вся специфика задаётся в
**настройках** — единственной редактируемой ячейке в начале файла.

**Как устроена работа.** Разделы 3–8 анализируют ряд и после каждого шага печатают проверку: ✓
если решение согласуется с данными, ⚠ с конкретной рекомендацией, если нет. Вы правите настройки
наверху и перезапускаете. Раздел 9 сравнивает несколько методов и предлагает лучший, раздел 10
считает итог, раздел 12 сохраняет настройки в JSON рядом с блокнотом.

**Зачем сохранять настройки.** Через полгода будет видно не только какие цифры получились, но и
почему они такие: какие выбросы приняты и на каком основании, почему выбран именно этот метод.
Без этого сезонная корректировка превращается в чёрный ящик, который невозможно ни проверить, ни
воспроизвести.

**Данные** берутся из файлов Excel рядом с блокнотом: колонки `Date` и `Value`, даты на конец
периода. В комплекте лежат денежная масса M2 и производственный календарь России — замените их
своими.
""",
"en": r"""
# Seasonal adjustment of an indicator: a working template

One indicator, one notebook. Copy it for each new series; everything specific is set in the
**settings** — the single editable cell at the top of the file.

**How it works.** Sections 3–8 analyse the series and after each step print a check: ✓ if the
decision agrees with the data, ⚠ with a concrete recommendation if it does not. You edit the
settings at the top and re-run. Section 9 compares several methods and suggests the best one,
section 10 computes the result, section 12 saves the settings as JSON next to the notebook.

**Why save the settings.** In six months it will be clear not only what the numbers were but why:
which outliers were accepted and on what grounds, why this particular method was chosen. Without
that, seasonal adjustment becomes a black box that can neither be checked nor reproduced.

**The data** come from Excel files next to the notebook: columns `Date` and `Value`, dates at the
end of the period. Russian money supply M2 and the Russian working calendar are included — replace
them with your own.
""",
}
show_md(MD)


In [ ]:
MD = {
"ru": r"""
## 1. Настройки показателя

Здесь принимаются все решения. Ячейки ниже ничего не задают сами — они только читают эти
настройки и проверяют, согласуются ли они с данными.

**Тип показателя** задаёт разумные стартовые значения: для потока в текущих ценах — логарифм и
рабочие дни, для запаса на дату — логарифм без календаря потока, для сальдо — без логарифма вовсе.
Дальше вы их подтверждаете или меняете по результатам разделов 6–8.

**Каждое решение с обоснованием.** Блок `justification` — это не формальность: он сохраняется
вместе с настройками, и через год объяснит, почему выбросы именно такие.
""",
"en": r"""
## 1. Indicator settings

All the decisions are made here. The cells below set nothing on their own — they only read these
settings and check whether they agree with the data.

**The indicator type** supplies sensible starting values: for a flow in current prices — a
logarithm and trading days, for a stock at a date — a logarithm without the flow calendar, for a
balance — no logarithm at all. You then confirm or change them based on sections 6–8.

**Every decision with a reason.** The `justification` block is not a formality: it is saved together
with the settings and a year from now will explain why the outliers are what they are.
""",
}
show_md(MD)


In [ ]:
TICKER   = "RU_M2_M_CB"              # имя файла с данными: <TICKER>.xlsx | data file name
CALENDAR = "RU_TDAYSWKPC_M_GV"       # производственный календарь, None если не нужен | working calendar
DATA_DIR = "."                       # папка с файлами | folder with the files

SETTINGS = new_passport(
    indicator_id   = TICKER,
    name_ru        = "Денежная масса М2",
    name_en        = "Money supply M2",
    source         = "Bank of Russia",        # свободный текст | free text
    units          = "RUB bn",                # свободный текст | free text
    frequency      = "M",
    indicator_type = "stock",        # flow_value | flow_volume | price_index | stock | balance | rate
    input_format   = "level",        # level | mom | qoq | ytd_cumulative
    table          = TICKER,
    engine         = "x11",
)

S = SETTINGS["spec"]

# --- выборка | sample ---------------------------------------------------
S["sample"]["start"] = "2008-01-31"   # раньше нет календаря | no calendar before that
S["sample"]["end"]   = None
# Обучающая выборка: на ней ОЦЕНИВАЕТСЯ модель, корректируется весь ряд (раздел 7.1).
# Model span: the model is ESTIMATED on it, the whole series is adjusted (section 7.1).
S["sample"]["model_span"] = {"start": None, "end": None, "origin": None, "reason": ""}

# --- модель | model -----------------------------------------------------
S["transform"] = "log"               # auto | log | none   (проверка -- раздел 6 | check -- section 6)

# --- календарь | calendar -----------------------------------------------
# kind="flow"  -- сколько рабочих дней было В месяце (выпуск, розница, перевозки)
# kind="stock" -- на какой день недели пришлась отчётная дата (остатки, резервы)
S["preprocess"]["calendar"]["trading_day"] = {
    "use": False, "kind": "stock", "stock_day": 31,
    "calendar_table": CALENDAR, "fallback_simple": True,
}
S["preprocess"]["calendar"]["easter"] = {"use": False, "tradition": "orthodox", "window": 8}

# --- выбросы | outliers -------------------------------------------------
# Типы: AO -- разовый всплеск, LS -- сдвиг уровня, TC -- затухающий всплеск,
#       SO -- сезонный выброс (только вручную, X-13 их не ищет; раздел 8.1).
# mode: auto -- искать и применять, manual -- только список ниже, none -- не учитывать.
S["preprocess"]["outliers"]["mode"] = "manual"
S["preprocess"]["outliers"]["list"] = [
    # Обоснование -- свободный текст на вашем языке | reason -- free text in your language
    {"date": "2008-10-31", "type": "LS", "reason": "global financial crisis"},
    {"date": "2009-01-31", "type": "LS", "reason": "crisis contraction of the money supply"},
]

# --- декомпозиция и выход | decomposition and output --------------------
S["decompose"]["engine"] = "x11"     # x11 | ucm | stl | x13_real  (выбор -- раздел 9 | section 9)
S["decompose"]["period"] = 12
S["output"]["saar_mode"] = "level_ma"          # level_ma | level_3m | rate_avg
S["output"]["forecast"]["horizon"] = 12
S["output"]["forecast_ci"] = 0.95

# --- обоснования | reasons ----------------------------------------------
SETTINGS["justification"].update({
    "sample":    "the working calendar starts in 2008",
    "transform": "",
    "calendar":  "",
    "method":    "",
})

err, warn = validate_passport(SETTINGS)
print(passport_summary(SETTINGS))
print()
for x in err:
    print("ОШИБКА / ERROR:", x)
for x in warn:
    print("!", x)
if not err:
    print(tx({"ru": "Настройки корректны.", "en": "The settings are valid."}))


In [ ]:
MD = {
"ru": r"""
## 2. Данные

Ряд и календарь читаются из Excel рядом с блокнотом. Сразу проверяется качество: пропуски, дубли
дат, даты не на конце периода, разрывы, длина ряда.

Последнее важнее, чем кажется: для сезонной корректировки нужно минимум три года, комфортно —
семь и больше. На коротком ряде сезонный профиль оценить нечем, и любой метод даст скорее шум,
чем сезонность.
""",
"en": r"""
## 2. The data

The series and the calendar are read from Excel next to the notebook. The quality is checked at
once: missing values, duplicate dates, dates not at the end of the period, gaps, length of the
series.

The last one matters more than it seems: seasonal adjustment needs at least three years, and seven
or more is comfortable. On a short series there is nothing to estimate the seasonal profile from,
and any method will produce noise rather than seasonality.
""",
}
show_md(MD)


In [ ]:
TXT = {
    "sec":   {"ru": "Данные", "en": "The data"},
    "head":  {"ru": "{0}: {1} .. {2}, {3} наблюдений, от {4:,.1f} до {5:,.1f} {6}",
              "en": "{0}: {1} .. {2}, {3} observations, from {4:,.1f} to {5:,.1f} {6}"},
    "na":    {"ru": "пропуски в значениях", "en": "missing values"},
    "dup":   {"ru": "дубли дат", "en": "duplicate dates"},
    "eom":   {"ru": "даты не на конце периода", "en": "dates not at the period end"},
    "gap":   {"ru": "пропущенные месяцы", "en": "missing months"},
    "len":   {"ru": "длина ряда, лет", "en": "length of the series, years"},
    "short": {"ru": "! ряд короче трёх лет -- корректировка ненадёжна",
              "en": "! the series is shorter than three years -- adjustment is unreliable"},
    "cal":   {"ru": "календарь: {0} .. {1}; не покрыто наблюдений ряда: {2}",
              "en": "calendar: {0} .. {1}; observations not covered: {2}"},
    "sample":{"ru": "выборка: {0} .. {1}, {2} наблюдений",
              "en": "sample: {0} .. {1}, {2} observations"},
}
set_section("2", TXT["sec"])

ind = SETTINGS["indicator"]
raw = load_indicator(TICKER, path=os.path.join(DATA_DIR, f"{TICKER}.xlsx"))
CAL = (load_working_days(CALENDAR, path=os.path.join(DATA_DIR, f"{CALENDAR}.xlsx"))
       if CALENDAR else None)

print(tx(TXT["head"]).format(TICKER, raw.index[0].date(), raw.index[-1].date(), len(raw),
                             raw.min(), raw.max(), ind["units"]))

idx = pd.DatetimeIndex(raw.index)
gaps = pd.Series(idx).diff().dt.days.dropna()
checks = [(tx(TXT["na"]), int(raw.isna().sum())),
          (tx(TXT["dup"]), int(idx.duplicated().sum())),
          (tx(TXT["eom"]), int((idx != idx + pd.offsets.MonthEnd(0)).sum())),
          (tx(TXT["gap"]), int((gaps > 35).sum())),
          (tx(TXT["len"]), round(len(raw) / 12, 1))]
for k, v in checks:
    print(f"  {k:32s}: {v}")
if len(raw) < 36:
    print(tx(TXT["short"]))

if CAL:
    cov = calendar_coverage(raw, CAL)
    print(tx(TXT["cal"]).format(cov["calendar_from"], cov["calendar_to"], cov["n_missing"]))

s_raw = raw.copy()
if S["sample"]["start"]:
    s_raw = s_raw[s_raw.index >= pd.Timestamp(S["sample"]["start"])]
if S["sample"]["end"]:
    s_raw = s_raw[s_raw.index <= pd.Timestamp(S["sample"]["end"])]
SETTINGS["run"]["data_cutoff"] = str(pd.Timestamp(s_raw.index[-1]).date())
print(tx(TXT["sample"]).format(s_raw.index[0].date(), s_raw.index[-1].date(), len(s_raw)))


In [ ]:
MD = {
"ru": r"""
## 3. Приведение к уровню и первый взгляд

Если данные приходят темпами или накопленным итогом, всё считается **на уровне**: у ряда темпов
сдвиг уровня неотличим от разового всплеска, и типизация выбросов ломается.

Три графика. Сам ряд, год к году и подряды по месяцам — последний самый информативный. Каждая
линия на нём — один календарный месяц по годам: если линии расходятся, сезонность есть; если
сближаются — слабеет; если одна линия скачком переходит на другой уровень и там остаётся, значит
сломался сезонный профиль этого месяца.
""",
"en": r"""
## 3. Converting to a level and a first look

If the data arrive as rates or as a cumulative total, everything is computed **on the level**: in a
series of rates a level shift is indistinguishable from a one-off spike, and the typing of outliers
breaks down.

Three charts. The series itself, the year-on-year rate, and the month sub-series — the last is the
most informative. Each line on it is one calendar month across years: if the lines spread apart
there is seasonality; if they converge it is weakening; if one line steps to a different level and
stays there, the seasonal profile of that month has broken.
""",
}
show_md(MD)


In [ ]:
TXT = {
    "sec":  {"ru": "Ряд и первый взгляд", "en": "The series and a first look"},
    "raw":  {"ru": "{0}: исходный ряд", "en": "{0}: the original series"},
    "yoy":  {"ru": "Год к году", "en": "Year on year"},
    "pct":  {"ru": "%", "en": "%"},
}
set_section("3", TXT["sec"])

level = pd.Series(convert_to_level(s_raw.values, ind["input_format"], dates=s_raw.index),
                  index=s_raw.index, name=TICKER)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(level.index, level.values, color=pal["lazur"])
chart_frame(axes[0], tx(TXT["raw"]).format(tx(ind["name"])), ind["units"], pal=pal)
yoy = 100 * (level / level.shift(12) - 1)
axes[1].plot(yoy.index, yoy.values, color=pal["zoloto"])
axes[1].axhline(0, color=pal["muted"], linewidth=0.7)
chart_frame(axes[1], tx(TXT["yoy"]), tx(TXT["pct"]), pal=pal)
plt.tight_layout(); plt.show()

plot_month_subseries(level, period=S["decompose"]["period"], transform=S["transform"])
plt.show()


In [ ]:
MD = {
"ru": r"""
## 4. Есть ли сезонность вообще

Если сезонности нет, ряд корректировать не нужно: процедура только добавит шума. Такое бывает,
например, с некоторыми финансовыми запасами.

Тест считается по SI-отношениям — ряд минус центрированное скользящее среднее — и, для быстро
растущего ряда, по логарифму. Иначе кривизна тренда съедает весь сигнал и тест рапортует
«сезонности нет» там, где она очевидна.
""",
"en": r"""
## 4. Is there any seasonality at all

If there is none, the series must not be adjusted: the procedure would only add noise. That happens,
for example, with some financial stocks.

The test runs on SI ratios — the series minus a centred moving average — and, for a fast-growing
series, on the logarithm. Otherwise the curvature of the trend eats the whole signal and the test
reports "no seasonality" where it is obvious.
""",
}
show_md(MD)


In [ ]:
TXT = {
    "sec":  {"ru": "Есть ли сезонность", "en": "Is there seasonality"},
    "scale":{"ru": "шкала теста: {0}", "en": "test scale: {0}"},
    "f":    {"ru": "F-тест на сезонные дамми : F = {0:.1f},  p = {1:.2e}",
             "en": "F-test on seasonal dummies : F = {0:.1f},  p = {1:.2e}"},
    "acf":  {"ru": "автокорреляция на лаге {0}: {1:+.2f}",
             "en": "autocorrelation at lag {0}: {1:+.2f}"},
    "pw":   {"ru": "доля сезонной мощности   : {0:.3f}",
             "en": "share of seasonal power   : {0:.3f}"},
    "none": {"ru": "! Сезонность не обнаружена. Рассмотрите engine='none' и публикуйте исходный ряд.",
             "en": "! No seasonality detected. Consider engine='none' and publish the original series."},
}
set_section("4", TXT["sec"])

hs = has_seasonality(level.values, level.index, period=S["decompose"]["period"])
print(tx(TXT["scale"]).format(hs["transform_used"]))
print(tx(TXT["f"]).format(hs["F"], hs["p_value"]))
print(tx(TXT["acf"]).format(S["decompose"]["period"], hs["acf_seasonal_lag"]))
print(tx(TXT["pw"]).format(hs["power_share"]))
print()
print(hs["comment"])
if not hs["has_seasonality"]:
    print(tx(TXT["none"]))

ev = plot_seasonality_evidence(level, period=S["decompose"]["period"], transform=S["transform"])
plt.show()


In [ ]:
MD = {
"ru": r"""
## 5. Логарифм или нет

Если сезонный размах растёт вместе с уровнем, нужен логарифм. Забыть про него опаснее, чем
кажется: остаточная сезонность будет нарастать к правому краю выборки — ровно там, где смотрят на
свежие данные.

Тест режет ряд на годы и проверяет, растёт ли размах внутри года вместе с уровнем года. Наклон
около 1 означает мультипликативность, около 0 — аддитивность. Ряды с нулями и отрицательными
значениями логарифмировать нельзя.
""",
"en": r"""
## 5. Logarithm or not

If the seasonal swing grows with the level, a logarithm is needed. Forgetting it is more dangerous
than it seems: residual seasonality will build up towards the right edge of the sample — exactly
where fresh data are read.

The test cuts the series into years and checks whether the swing inside a year grows with the level
of that year. A slope near 1 means multiplicative, near 0 additive. Series with zeros and negative
values cannot be logged.
""",
}
show_md(MD)


In [ ]:
TXT = {
    "sec":  {"ru": "Логарифм или нет", "en": "Logarithm or not"},
    "res":  {"ru": "тест «размах против уровня»: наклон {0:.2f} -> рекомендуется transform = '{1}'",
             "en": "the swing-versus-level test: slope {0:.2f} -> recommended transform = '{1}'"},
    "na":   {"ru": "тест неприменим ({0}) -> рекомендуется transform = '{1}'",
             "en": "the test does not apply ({0}) -> recommended transform = '{1}'"},
    "cur":  {"ru": "в настройках: transform = '{0}'", "en": "in the settings: transform = '{0}'"},
    "ok":   {"ru": "OK настройки согласуются с тестом", "en": "OK the settings agree with the test"},
    "bad":  {"ru": "! расходится с тестом. Либо поставьте '{0}', либо обоснуйте экономически: "
                   "ряды в текущих ценах -- логарифм, сальдо -- никогда.",
             "en": "! disagrees with the test. Either set '{0}' or justify it economically: series in "
                   "current prices take a logarithm, balances never do."},
}
set_section("5", TXT["sec"])

choice, slope, why = choose_transform(level.values, return_detail=True)
if np.isfinite(slope):
    print(tx(TXT["res"]).format(slope, choice))
else:
    print(tx(TXT["na"]).format(why, choice))
print(tx(TXT["cur"]).format(S["transform"]))
print(tx(TXT["ok"]) if S["transform"] in (choice, "auto") else tx(TXT["bad"]).format(choice))


In [ ]:
MD = {
"ru": r"""
## 6. Календарь: поток и запас проверяются по-разному

Рабочие дни и Пасха — два разных эффекта, и решение по ним принимается раздельно. **Пасха не
входит в производственный календарь**: она всегда приходится на воскресенье и число рабочих дней
не меняет; её эффект — сдвиг спроса между месяцами.

Сами рабочие дни тоже бывают двух видов. Для **потока** — выпуска, розницы, перевозок — важно,
сколько рабочих дней было в месяце. Для **запаса** — денежной массы, остатков, резервов — это
неважно: значение снимается на одну дату, и важно, на какой день недели она пришлась. В X-13 для
этого отдельный регрессор `tdstock[w]`, где `31` означает «последний день месяца».

Проверяются они по-разному: у потока один коэффициент и t-статистика, у запаса шесть столбцов дня
недели и совместный F-тест. И осторожно с выводом «значим — значит подходит»: регрессоры
коррелируют, поэтому на ряде-запасе потоковый тоже часто значим. Решает не значимость, а качество
подгонки — AICC.
""",
"en": r"""
## 6. The calendar: flows and stocks are tested differently

Working days and Easter are two different effects, decided separately. **Easter is not part of the
working calendar**: it always falls on a Sunday and does not change the number of working days; its
effect is a shift of demand between months.

Working days themselves come in two kinds. For a **flow** — output, retail, freight — what matters
is how many working days there were in the month. For a **stock** — money supply, balances,
reserves — that is irrelevant: the value is taken on one date, and what matters is which day of the
week it fell on. X-13 has a separate regressor for this, `tdstock[w]`, where `31` means "the last
day of the month".

They are tested differently: a flow is one coefficient and a t-statistic, a stock is six weekday
columns and a joint F-test. And beware of "significant means appropriate": the regressors correlate,
so on a stock series the flow one often looks significant too. What decides is not significance but
the fit — AICC.
""",
}
show_md(MD)


In [ ]:
TXT = {
    "sec":  {"ru": "Календарь", "en": "The calendar"},
    "kind": {"ru": "тип календаря по настройкам: {0}{1}",
             "en": "calendar kind from the settings: {0}{1}"},
    "day":  {"ru": " (день фиксации {0})", "en": " (recording day {0})"},
    "ok":   {"ru": "OK {0:14s}: тест говорит «{1}» ({2}, AICC {3:+.2f}), в настройках use={4}",
             "en": "OK {0:14s}: the test says '{1}' ({2}, AICC {3:+.2f}), settings use={4}"},
    "bad":  {"ru": "!  {0:14s}: тест говорит «{1}» ({2}, AICC {3:+.2f}), в настройках use={4}",
             "en": "!  {0:14s}: the test says '{1}' ({2}, AICC {3:+.2f}), settings use={4}"},
    "other":{"ru": "\nДругой вариант регрессора -- для сравнения:",
             "en": "\nThe other regressor variant, for comparison:"},
}
set_section("6", TXT["sec"])

TD_KIND = S["preprocess"]["calendar"]["trading_day"].get("kind", "flow")
STOCK_DAY = S["preprocess"]["calendar"]["trading_day"].get("stock_day", 31)
z = np.log(level.values) if S["transform"] == "log" else level.values

cal_tbl = calendar_significance_ext(
    z, level.index, period=S["decompose"]["period"], working_days_calendar=CAL,
    kind=TD_KIND, stock_day=STOCK_DAY,
    easter_tradition=S["preprocess"]["calendar"]["easter"]["tradition"],
    easter_window=S["preprocess"]["calendar"]["easter"]["window"])

print(tx(TXT["kind"]).format(TD_KIND, tx(TXT["day"]).format(STOCK_DAY) if TD_KIND == "stock" else ""))
print(cal_tbl.round(3).to_string(index=False))
print()
for _, r in cal_tbl.iterrows():
    key = "easter" if r["regressor"] == "easter" else "trading_day"
    in_set = S["preprocess"]["calendar"][key]["use"]
    rec = r["verdict"] in ("включать", "include")
    stat = (f"t={r['t_stat']:.2f}" if np.isfinite(r.get("t_stat", np.nan))
            else f"F={r['F']:.2f}, p={r['p_value']:.3f}")
    line = TXT["ok"] if in_set == rec else TXT["bad"]
    print(tx(line).format(str(r["regressor"]), r["verdict"], stat, r["aicc_delta"], in_set))

other = "flow" if TD_KIND == "stock" else "stock"
print(tx(TXT["other"]))
print(calendar_significance_ext(z, level.index, working_days_calendar=CAL, kind=other,
                                stock_day=STOCK_DAY).round(3).to_string(index=False))


In [ ]:
MD = {
"ru": r"""
### 6.1. Обучающая выборка: на чём оцениваем модель

Кризис в начале ряда или методологический разрыв искажает оценку сезонного профиля на все
последующие годы. Вырезать такой кусок целиком — потерять данные; оставить в оценивании — исказить
сегодняшние факторы. X-13 решает это блоком `modelspan`: модель оценивается на выбранном
интервале, а корректировка применяется ко **всему** ряду.

Подсказка ниже считается так: для каждого кандидата на начало ряд раскладывается, берутся сезонные
факторы последних лет и сравниваются с оценкой по самому короткому окну. Рекомендуется самое раннее
начало, при котором сегодняшние факторы ещё не смещаются больше допуска.

Это подсказка, а не приговор: решение экспертное и должно попасть в настройки с обоснованием.
Конец интервала часто имеет смысл сдвинуть назад: самые свежие наблюдения ещё предварительны.
Обучающая выборка работает только с движком `x13_real`.
""",
"en": r"""
### 6.1. The model span: what the model is estimated on

A crisis at the start of a series or a methodological break distorts the estimate of the seasonal
profile for all the years that follow. Cutting such a piece out altogether loses data; leaving it in
the estimation distorts today's factors. X-13 solves this with `modelspan`: the model is estimated
on the chosen interval and the adjustment is applied to the **whole** series.

The hint below is computed as follows: for every candidate start the series is decomposed, the
seasonal factors of the recent years are taken and compared with the estimate from the shortest
window. The recommendation is the earliest start at which today's factors do not yet shift by more
than the tolerance.

It is a hint, not a verdict: the choice is an expert one and should go into the settings with a
reason. The end of the interval is often worth moving back: the freshest observations are still
preliminary. The model span works only with the `x13_real` engine.
""",
}
show_md(MD)


In [ ]:
TXT = {
    "sec":  {"ru": "Обучающая выборка", "en": "The model span"},
    "tol":  {"ru": "допуск по сдвигу сезонных факторов: {0:.3f}",
             "en": "tolerance for the shift of seasonal factors: {0:.3f}"},
    "rec":  {"ru": "\nрасчётная рекомендация начала: {0}",
             "en": "\ncomputed recommendation for the start: {0}"},
    "cur":  {"ru": "в настройках: {0} .. {1} ({2}) -- {3}",
             "en": "in the settings: {0} .. {1} ({2}) -- {3}"},
    "none": {"ru": "в настройках обучающая выборка не задана -- модель оценивается по всей выборке",
             "en": "no model span in the settings -- the model is estimated on the whole sample"},
    "how":  {"ru": "чтобы применить рекомендацию, впишите в настройки:",
             "en": "to apply the recommendation, put this in the settings:"},
    "only": {"ru": "\nВажно: обучающая выборка поддерживается только движком x13_real.",
             "en": "\nNote: the model span is supported only by the x13_real engine."},
}
set_section("6.1", TXT["sec"])

span_tbl, span_start, span_tol = suggest_model_span(
    level, period=S["decompose"]["period"], transform=S["transform"])
print(tx(TXT["tol"]).format(span_tol))
print(span_tbl.round(3).to_string(index=False))
print(tx(TXT["rec"]).format(span_start))

MS = S["sample"].get("model_span") or {}
if MS.get("start") or MS.get("end"):
    print(tx(TXT["cur"]).format(MS.get("start"), MS.get("end"), MS.get("origin"), MS.get("reason")))
    plot_model_span(level, SETTINGS); plt.show()
else:
    print(tx(TXT["none"]))
    print(tx(TXT["how"]))
    print(f'    S["sample"]["model_span"] = {{"start": "{span_start}", "end": None,')
    print('                                 "origin": "auto", "reason": "..."}')
print(tx(TXT["only"]))


In [ ]:
MD = {
"ru": r"""
## 7. Выбросы

Алгоритм предлагает кандидатов, решение принимаете вы. Поиск идёт с открытым краем: кандидаты
последнего года помечены в колонке `edge`. Их решение предварительно — свежий шок нельзя надёжно
отличить от начала нового тренда, для этого нужны следующие наблюдения.

Колонка `event` показывает совпадение со справочником известных шоков российской экономики с
допуском в месяц. Ниже отдельно выводятся события, на которые алгоритм **не** отреагировал: иногда
это правильно, а иногда подсказка, что порог слишком строгий. Свой справочник передаётся
аргументом `events=`.
""",
"en": r"""
## 7. Outliers

The algorithm proposes candidates, you make the decision. The search runs with an open end:
candidates from the last year are marked in the `edge` column. Their decision is provisional — a
fresh shock cannot be reliably told from the start of a new trend, that needs the next observations.

The `event` column shows a match with the reference book of known shocks to the Russian economy,
with a tolerance of one month. Below, separately, are the events the algorithm did **not** react to:
sometimes that is right, and sometimes a hint that the threshold is too strict. Your own reference
book is passed with the `events=` argument.
""",
}
show_md(MD)


In [ ]:
TXT = {
    "sec":   {"ru": "Выбросы", "en": "Outliers"},
    "cand":  {"ru": "Кандидаты в выбросы:", "en": "Candidate outliers:"},
    "drop":  {"ru": "отсеяно обратным проходом: {0}", "en": "dropped by the backward pass: {0}"},
    "miss":  {"ru": "\nСобытия, на которые алгоритм не отреагировал:",
              "en": "\nEvents the algorithm did not react to:"},
    "only":  {"ru": "\nЗадано в настройках, но алгоритм не нашёл (нормально, если основание содержательное):",
              "en": "\nSet in the settings but not found by the algorithm (fine if the reason is substantive):"},
    "inset": {"ru": "в настройках", "en": "in settings"},
    "yes":   {"ru": "да", "en": "yes"},
    "chart": {"ru": "Выбросы, принятые в настройки", "en": "Outliers accepted into the settings"},
    "sub":   {"ru": "вертикальные линии -- даты из списка", "en": "vertical lines -- dates from the list"},
}
set_section("7", TXT["sec"])

z_series = pd.Series(z, index=level.index)
cand = outlier_candidates(z_series, period=S["decompose"]["period"],
                          cv=S["preprocess"]["outliers"]["cv"],
                          types=S["preprocess"]["outliers"]["types"],
                          skip_end=0, gls=S["preprocess"]["outliers"]["gls"],
                          scale="log" if S["transform"] == "log" else "level")

in_set = {(o["date"], o["type"]) for o in S["preprocess"]["outliers"]["list"]}
cand[tx(TXT["inset"])] = [(tx(TXT["yes"]) if (r.date, r.type) in in_set else "") for r in cand.itertuples()]
print(tx(TXT["cand"]))
print(cand[["date", "type", "t_stat", "effect_pct", "edge", "event", tx(TXT["inset"])]]
      .round(2).to_string(index=False))
if cand.attrs.get("dropped_backward"):
    print(tx(TXT["drop"]).format(cand.attrs["dropped_backward"]))

miss = events_without_candidates(level, cand)
if len(miss):
    print(tx(TXT["miss"]))
    print(miss.to_string(index=False))

in_cand = {(r.date, r.type) for r in cand.itertuples()}
only_set = [o for o in S["preprocess"]["outliers"]["list"] if (o["date"], o["type"]) not in in_cand]
if only_set:
    print(tx(TXT["only"]))
    for o in only_set:
        print(f"   {o['date']} {o['type']}  {o.get('reason', '')}")

fig, ax = plt.subplots(figsize=(12, 4.6))
ax.plot(level.index, level.values, color=pal["lazur"])
for o in S["preprocess"]["outliers"]["list"]:
    d = pd.Timestamp(o["date"])
    ax.axvline(d, color=pal["zoloto"], linestyle="--", linewidth=1.2)
    ax.annotate(o["type"], (d, level.max()), fontsize=8, color=pal["zoloto"],
                xytext=(2, -10), textcoords="offset points")
chart_frame(ax, tx(TXT["chart"]), tx(TXT["sub"]), pal=pal)
plt.tight_layout(); plt.show()


In [ ]:
MD = {
"ru": r"""
### 7.1. Сезонные выбросы SO

Сезонный выброс — это когда с некоторой даты меняется сезонный профиль **одного календарного
месяца** и остаётся таким навсегда: например, май много лет был выше тренда, а потом ушёл ниже.
Уровень ряда при этом не сдвигается, меняется распределение внутри года, поэтому ни AO, ни LS, ни
TC такое не описывают.

**X-13 сезонные выбросы не ищет** — автопоиск принимает только AO, LS и TC. Оценить заданный SO
движок умеет, а найти его должен аналитик. Поиск ниже режет ряд по календарным месяцам и внутри
каждого ищет слом среднего по годам. На графике ступенька на одной линии — это и есть кандидат.

SO применяется только движком `x13_real`; при `engine="x11"` он будет проигнорирован.
""",
"en": r"""
### 7.1. Seasonal outliers, SO

A seasonal outlier is when from some date the seasonal profile of **one calendar month** changes and
stays changed: for years May was above the trend, and then it moved below. The level of the series
does not shift, the distribution within the year does, so neither AO nor LS nor TC describes it.

**X-13 does not search for seasonal outliers** — its automatic search accepts only AO, LS and TC. The
engine can estimate an SO that is given to it, but finding one is the analyst's job. The search below
cuts the series by calendar month and inside each looks for a break in the mean across years. On the
chart, a step in one line is the candidate.

SO is applied only by the `x13_real` engine; with `engine="x11"` it is ignored.
""",
}
show_md(MD)


In [ ]:
TXT = {
    "sec":  {"ru": "Сезонные выбросы", "en": "Seasonal outliers"},
    "found":{"ru": "Кандидаты в сезонные выбросы:", "en": "Candidate seasonal outliers:"},
    "none": {"ru": "Кандидатов в сезонные выбросы не найдено.",
             "en": "No candidate seasonal outliers found."},
    "have": {"ru": "\nв настройках SO: {0}", "en": "\nSO in the settings: {0}"},
    "how":  {"ru": "Чтобы принять кандидата, добавьте в список выбросов, например:",
             "en": "To accept a candidate, add it to the outlier list, for example:"},
}
set_section("7.1", TXT["sec"])

so_tbl = so_candidates(level, period=S["decompose"]["period"], transform=S["transform"], cv=3.0)
if len(so_tbl):
    print(tx(TXT["found"]))
    print(so_tbl[["date", "month", "t_stat", "shift_pp", "years_before", "years_after"]]
          .round(2).to_string(index=False))
else:
    print(tx(TXT["none"]))

fig, _ = plot_so_candidates(level, so_tbl, SETTINGS, period=S["decompose"]["period"],
                            transform=S["transform"])
plt.show()

so_in = [o for o in S["preprocess"]["outliers"]["list"] if o.get("type") == "SO"]
print(tx(TXT["have"]).format(len(so_in)))
for o in so_in:
    print(f"   {o['date']}  {o.get('reason', '')}")
if len(so_tbl) and not so_in:
    r = so_tbl.iloc[0]
    print(tx(TXT["how"]))
    print(f'    {{"date": "{r["date"]}", "type": "SO", "reason": "..."}}')


In [ ]:
MD = {
"ru": r"""
## 8. Сравнение методов и выбор

На реальных данных истинной сезонности не существует, поэтому сравнивать методы по ошибке
относительно неё нельзя. Таблица двухъярусная.

**Ярус «годен»** — нет остаточной сезонности по F-тесту и сезонная мощность упала минимум вдвое.
Обе метрики считаются по детрендированным рядам: у сильно трендового ряда спектр забит трендом, и
абсолютная доля перестаёт что-либо различать.

**Ранжирование среди годных** — величина пересмотров (ряд обрезается, спецификация применяется
заново, сравниваются оценки тех же месяцев) и шумность SA-ряда. Для публикации пересмотры важнее:
цифра, которая через два месяца станет другой, хуже чуть более шумной, но стабильной.

Список движков по умолчанию — `x11`, `ucm`, `stl` плюс `x13_real`, если бинарник найден. Учтите
время: метрика пересмотров прогоняет спецификацию на обрезанных выборках.
""",
"en": r"""
## 8. Comparing the methods and choosing

On real data the true seasonality does not exist, so methods cannot be compared by the error against
it. The table has two tiers.

**The "fit for use" tier** — no residual seasonality by the F-test and the seasonal power at least
halved. Both metrics are computed on detrended series: in a strongly trending series the spectrum is
dominated by the trend and the absolute share stops distinguishing anything.

**Ranking among those fit** — the size of revisions (the series is truncated, the specification is
applied again, the estimates of the same months are compared) and the noisiness of the SA series. For
a publication revisions matter more: a figure that will be different in two months is worse than a
slightly noisier but stable one.

The default engine list is `x11`, `ucm`, `stl` plus `x13_real` if the binary is found. Mind the time:
the revision metric re-runs the specification on truncated samples.
""",
}
show_md(MD)


In [ ]:
TXT = {
    "sec":  {"ru": "Сравнение методов", "en": "Comparing the methods"},
    "bin":  {"ru": "бинарник x13as: {0}", "en": "x13as binary: {0}"},
    "yes":  {"ru": "доступен", "en": "available"},
    "no":   {"ru": "не найден (x13_real пропущен)", "en": "not found (x13_real skipped)"},
    "rec":  {"ru": "\nрекомендуется: {0}", "en": "\nrecommended: {0}"},
    "cur":  {"ru": "в настройках : {0}", "en": "in the settings: {0}"},
    "ok":   {"ru": "OK настройки согласуются с рекомендацией",
             "en": "OK the settings agree with the recommendation"},
    "bad":  {"ru": "! рекомендация расходится с выбором. Либо поставьте '{0}', либо запишите причину "
                   "в justification.method.",
             "en": "! the recommendation differs from the choice. Either set '{0}' or record the reason "
                   "in justification.method."},
    "offic":{"ru": "\nx13_real -- официальный движок; если он прошёл первый ярус, для публикации "
                   "ориентируйтесь на него.",
             "en": "\nx13_real is the official engine; if it passed the first tier, use it for publication."},
}
set_section("8", TXT["sec"])

print(tx(TXT["bin"]).format(tx(TXT["yes"]) if x13_available() else tx(TXT["no"])))
# already_level=True: ряд приведён к уровню выше | converted to a level above
tbl, best, _ = compare_methods(level, SETTINGS, working_days_calendar=CAL,
                               already_level=True)
print(tbl.round(3).to_string(index=False))
print(tx(TXT["rec"]).format(best))
print(tx(TXT["cur"]).format(S["decompose"]["engine"]))
if best and S["decompose"]["engine"] != best:
    print(tx(TXT["bad"]).format(best))
else:
    print(tx(TXT["ok"]))
if "x13_real" in set(tbl["engine"]):
    print(tx(TXT["offic"]))


In [ ]:
MD = {
"ru": r"""
## 9. Итоговый расчёт

Настройки применяются целиком: та же функция, что использовалась при сравнении методов, поэтому
итог совпадает со строкой выбранного движка в таблице выше.
""",
"en": r"""
## 9. The final calculation

The settings are applied in full by the same function that was used in the comparison, so the result
matches the row of the chosen engine in the table above.
""",
}
show_md(MD)


In [ ]:
TXT = {
    "sec":  {"ru": "Итоговый расчёт", "en": "The final calculation"},
    "mod":  {"ru": "модель: {0} | движок: {1}", "en": "model: {0} | engine: {1}"},
    "out":  {"ru": "выбросы применены: {0}", "en": "outliers applied: {0}"},
    "none": {"ru": "нет", "en": "none"},
    "bt":   {"ru": "бэктест SA-ряда: RMSE={0:.1f}, MAPE={1:.2f}%",
             "en": "backtest of the SA series: RMSE={0:.1f}, MAPE={1:.2f}%"},
}
set_section("9", TXT["sec"])

RESULT = apply_passport(level, SETTINGS, working_days_calendar=CAL, saar_n_sim=1500,
                        already_level=True)
q = RESULT["quality"]
dates_r = pd.DatetimeIndex(RESULT["dates"])
sa = pd.Series(RESULT["decomposition"]["sa"], index=dates_r)
fc = RESULT["forecast"]

print(tx(TXT["mod"]).format(RESULT["transform"], S["decompose"]["engine"]))
print(tx(TXT["out"]).format(", ".join(f"{pd.Timestamp(dates_r[i]).date()} {k}"
                                      for i, k, _ in sorted(RESULT["outliers"])) or tx(TXT["none"])))
print()
print(q["interpretation"])
if RESULT.get("backtest"):
    print(tx(TXT["bt"]).format(RESULT["backtest"]["rmse"], RESULT["backtest"]["mape"]))
for w in RESULT.get("warnings", []):
    print("!", w)


In [ ]:
TXT = {
    "raw":   {"ru": "исходный", "en": "original"},
    "sa":    {"ru": "сезонно скорректированный", "en": "seasonally adjusted"},
    "fc":    {"ru": "прогноз", "en": "forecast"},
    "ci":    {"ru": "{0:.0%} интервал", "en": "{0:.0%} interval"},
    "t1":    {"ru": "{0}: сезонная корректировка и прогноз",
              "en": "{0}: seasonal adjustment and forecast"},
    "saar":  {"ru": "3MMA SAAR", "en": "3MMA SAAR"},
    "yoy":   {"ru": "год к году", "en": "year on year"},
    "fcy":   {"ru": "прогноз г/г", "en": "forecast y/y"},
    "t2":    {"ru": "Темпы роста: 3MMA SAAR и год к году",
              "en": "Growth rates: 3MMA SAAR and year on year"},
    "s2":    {"ru": "% годовых, конвенция {0}", "en": "% annualised, convention {0}"},
    "edge":  {"ru": "Край ряда ненадёжен: последние месяцы будут пересмотрены с новыми данными.",
              "en": "The end of the series is unreliable: the last months will be revised."},
}

SHOW_YOY = True


def label_last(ax, s, color, fmt="{:.1f}%"):
    s = s.dropna()
    if len(s):
        ax.plot([s.index[-1]], [s.iloc[-1]], "o", ms=4, color=color)
        ax.annotate(fmt.format(s.iloc[-1]), (s.index[-1], s.iloc[-1]), color=color, fontsize=9,
                    fontweight="bold", xytext=(6, 0), textcoords="offset points", va="center")


fut = pd.DatetimeIndex(fc["dates"]) if fc is not None else pd.DatetimeIndex([])
ci = fc.get("ci", 0.95) if fc is not None else 0.95

fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(level.index, level.values, color=pal["muted"], alpha=0.45, label=tx(TXT["raw"]))
ax.plot(sa.index, sa.values, color=pal["lazur"], linewidth=1.8, label=tx(TXT["sa"]))
if fc is not None:
    ax.plot(fut, fc["forecast"], "--", color=pal["korall"], label=tx(TXT["fc"]))
    if fc.get("lo") is not None:
        ax.fill_between(fut, fc["lo"], fc["hi"], color=pal["korall"], alpha=0.15,
                        label=tx(TXT["ci"]).format(ci))
chart_frame(ax, tx(TXT["t1"]).format(tx(ind["name"])), ind["units"], pal=pal)
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

saar = pd.Series(np.asarray(RESULT["saar_hist"]), index=dates_r)
lvl_all = pd.Series(RESULT["y_level"], index=dates_r)
if fc is not None:
    lvl_all = pd.concat([lvl_all, pd.Series(fc.get("forecast_original", fc["forecast"]), index=fut)])
yoy_all = 100 * (lvl_all / lvl_all.shift(12) - 1)
sf = RESULT.get("saar_forecast")
n_hist = len(dates_r)

fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(saar.index, saar.values, color=pal["lazur"], linewidth=1.6, label=tx(TXT["saar"]))
if sf is not None and fc is not None:
    ax.plot(fut, sf["point"][n_hist:], "--", color=pal["lazur"], alpha=0.8, label=tx(TXT["fc"]))
    ax.fill_between(fut, sf["lo"][n_hist:], sf["hi"][n_hist:], color=pal["lazur"], alpha=0.13,
                    label=tx(TXT["ci"]).format(sf.get("ci", ci)))
label_last(ax, saar, pal["lazur"])
if SHOW_YOY:
    yoy_fact = yoy_all[yoy_all.index <= dates_r[-1]]
    ax.plot(yoy_fact.index, yoy_fact.values, color=pal["volna"], linewidth=1.4, label=tx(TXT["yoy"]))
    if len(fut):
        yoy_fc = yoy_all[yoy_all.index > dates_r[-1]]
        ax.plot(yoy_fc.index, yoy_fc.values, "--", color=pal["volna"], alpha=0.8, label=tx(TXT["fcy"]))
    label_last(ax, yoy_fact, pal["volna"])
ax.axhline(0, color=pal["muted"], linewidth=0.7)
chart_frame(ax, tx(TXT["t2"]), tx(TXT["s2"]).format(S["output"]["saar_mode"]), pal=pal)
ax.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()

check = plot_seasonality_check(level.values, sa.values, level.index,
                               period=S["decompose"]["period"], transform=S["transform"])
plt.show()
print(tx(TXT["edge"]))


In [ ]:
MD = {
"ru": r"""
## 10. Сверка с официальной корректировкой (необязательно)

Если для вашего показателя кто-то публикует свою сезонно скорректированную версию — статистическая
служба, центральный банк, международная организация, — положите её файл рядом и укажите имя в
`COMPARE_WITH`. Это полезная проверка: она показывает, насколько ваш результат отличается от
общепринятого.

Большое расхождение — повод проверить три вещи в таком порядке: конвенцию (что именно публикует
источник), календарь и выбросы. И помните про реализацию: встроенные `-lite` методы проще настоящего X-13. На денежной массе
M2 они дают расхождение с оценкой Банка России 0,3–0,6% против 0,3% у `x13_real` — то есть
близко, но на другом ряде разрыв может оказаться больше. Для публикуемых цифр берите
официальный движок.
""",
"en": r"""
## 10. Comparison with an official adjustment (optional)

If someone publishes their own seasonally adjusted version of your indicator — a statistical office,
a central bank, an international organisation — put their file next to the notebook and give its name
in `COMPARE_WITH`. It is a useful check: it shows how far your result is from the accepted one.

A large discrepancy is a reason to check three things in this order: the convention (what exactly the
source publishes), the calendar and the outliers. And remember the implementation: the built-in `-lite` methods are simpler than the real X-13. On
Russian M2 they differ from the Bank of Russia estimate by 0.3–0.6% against 0.3% for
`x13_real` — close, but on another series the gap may be larger. For published figures use the
official engine.
""",
}
show_md(MD)


In [ ]:
COMPARE_WITH = None        # например "RU_M2SA_M_CB" -- файл <имя>.xlsx рядом с блокнотом
                           # e.g. "RU_M2SA_M_CB" -- the file <name>.xlsx next to the notebook

TXT = {
    "sec":  {"ru": "Сверка с официальной корректировкой", "en": "Comparison with an official adjustment"},
    "off":  {"ru": "не задана -- сверять не с чем", "en": "not set -- nothing to compare with"},
    "nof":  {"ru": "файл {0}.xlsx не найден", "en": "file {0}.xlsx not found"},
    "res":  {"ru": "совпадающих месяцев: {0} | среднее |отклонение| {1:.2f}% | максимум {2:.2f}%",
             "en": "months in common: {0} | mean |deviation| {1:.2f}% | maximum {2:.2f}%"},
    "corr": {"ru": "корреляция темпов м/м: {0:.3f}", "en": "correlation of m/m rates: {0:.3f}"},
    "t1":   {"ru": "Сезонно скорректированный ряд: наш и официальный",
             "en": "Seasonally adjusted series: ours and the official one"},
    "ours": {"ru": "наша оценка", "en": "our estimate"},
    "theirs":{"ru": "официальная", "en": "official"},
    "t2":   {"ru": "Расхождение", "en": "Discrepancy"},
}
set_section("10", TXT["sec"])

if not COMPARE_WITH:
    print(tx(TXT["off"]))
elif not os.path.exists(os.path.join(DATA_DIR, f"{COMPARE_WITH}.xlsx")):
    print(tx(TXT["nof"]).format(COMPARE_WITH))
else:
    off = load_indicator(COMPARE_WITH, path=os.path.join(DATA_DIR, f"{COMPARE_WITH}.xlsx"))
    common = sa.index.intersection(off.index)
    d = 100 * (sa.loc[common] / off.loc[common] - 1)
    print(tx(TXT["res"]).format(len(common), d.abs().mean(), d.abs().max()))
    print(tx(TXT["corr"]).format(
        float(np.corrcoef(sa.loc[common].pct_change().dropna(),
                          off.loc[common].pct_change().dropna())[0, 1])))

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                             gridspec_kw={"height_ratios": [2.2, 1]})
    axes[0].plot(common, sa.loc[common], color=pal["lazur"], label=tx(TXT["ours"]))
    axes[0].plot(common, off.loc[common], color=pal["zoloto"], label=tx(TXT["theirs"]))
    chart_frame(axes[0], tx(TXT["t1"]), ind["units"], pal=pal)
    axes[0].legend(fontsize=8)
    axes[1].plot(common, d.values, color=pal["korall"])
    axes[1].axhline(0, color=pal["muted"], linewidth=0.7)
    chart_frame(axes[1], tx(TXT["t2"]), "%", pal=pal)
    plt.tight_layout(); plt.show()


In [ ]:
MD = {
"ru": r"""
## 11. Сохранение настроек и результата

Настройки сохраняются в JSON рядом с блокнотом, рассчитанные ряды — в CSV. Файл настроек и есть
главная ценность: цифры всегда можно пересчитать, а вот восстановить по памяти, почему выбросом
признан именно март 2022 и почему выбран именно этот метод, через полгода уже не получится.
""",
"en": r"""
## 11. Saving the settings and the result

The settings are saved as JSON next to the notebook, the computed series as CSV. The settings file is
the valuable part: the numbers can always be recomputed, but recalling from memory why exactly March
2022 was marked as an outlier and why this method was chosen will not work six months later.
""",
}
show_md(MD)


In [ ]:
TXT = {
    "sec":  {"ru": "Сохранение", "en": "Saving"},
    "set":  {"ru": "настройки: {0}", "en": "settings: {0}"},
    "csv":  {"ru": "ряды:      {0}", "en": "series:    {0}"},
    "fcst": {"ru": "прогноз:   {0}", "en": "forecast:  {0}"},
    "load": {"ru": "\nПрочитать настройки обратно: SETTINGS = load_passport_file(\"{0}\")",
             "en": "\nRead the settings back: SETTINGS = load_passport_file(\"{0}\")"},
}
set_section("11", TXT["sec"])

path_settings = save_passport(SETTINGS)
print(tx(TXT["set"]).format(path_settings))

out = pd.DataFrame({"level": RESULT["y_level"], "sa": sa.values,
                    "saar": np.asarray(RESULT["saar_hist"])}, index=dates_r).rename_axis("Date")
csv_path = f"{TICKER}_series.csv"
out.to_csv(csv_path)
print(tx(TXT["csv"]).format(csv_path))

if fc is not None:
    fc_path = f"{TICKER}_forecast.csv"
    pd.DataFrame({"forecast": fc["forecast"], "lo": fc.get("lo"), "hi": fc.get("hi")},
                 index=fut).rename_axis("Date").to_csv(fc_path)
    print(tx(TXT["fcst"]).format(fc_path))

print(tx(TXT["load"]).format(path_settings))


In [ ]:
MD = {
"ru": r"""
## 12. Сводка

Ключевые цифры и то, что осталось открытым на следующий месяц.
""",
"en": r"""
## 12. Summary

The key figures and what remains open for next month.
""",
}
show_md(MD)


In [ ]:
TXT = {
    "sec":  {"ru": "Сводка", "en": "Summary"},
    "key":  {"ru": "КЛЮЧЕВЫЕ ЦИФРЫ", "en": "KEY FIGURES"},
    "last": {"ru": "  последнее наблюдение     : {0}  {1:,.1f} {2}",
             "en": "  last observation         : {0}  {1:,.1f} {2}"},
    "sa":   {"ru": "  сезонно скорректировано  : {0:,.1f}", "en": "  seasonally adjusted      : {0:,.1f}"},
    "saar": {"ru": "  3MMA SAAR, последний мес.: {0:.1f}% годовых",
             "en": "  3MMA SAAR, last month    : {0:.1f}% annualised"},
    "med":  {"ru": "  3MMA SAAR, медиана 3 года: {0:.1f}% годовых",
             "en": "  3MMA SAAR, 3-year median : {0:.1f}% annualised"},
    "open": {"ru": "\nОТКРЫТЫЕ ВОПРОСЫ", "en": "\nOPEN QUESTIONS"},
    "edge": {"ru": "  кандидаты у края ряда: {0} -- решение отложено до новых данных",
             "en": "  candidates at the end of the series: {0} -- decision deferred"},
    "noedge":{"ru": "  кандидатов у края ряда нет", "en": "  no candidates at the end of the series"},
    "rev":  {"ru": "  типичная величина пересмотра выбранного метода: {0:.2f}%",
             "en": "  typical revision of the chosen method: {0:.2f}%"},
    "skip": {"ru": "  раздел {0} в этой сессии не запускался -- запустите блокнот целиком",
             "en": "  section {0} was not run in this session -- run the whole notebook"},
    "norun":{"ru": "\nРасчёт в этой сессии не выполнялся: запустите блокнот целиком (Run All).",
             "en": "\nNo calculation in this session: run the whole notebook (Run All)."},
}
set_section("12", TXT["sec"])

print(passport_summary(SETTINGS))
if "RESULT" not in dir():
    print(tx(TXT["norun"]))
    raise SystemExit
print()
print(tx(TXT["key"]))
print(tx(TXT["last"]).format(level.index[-1].date(), level.iloc[-1], ind["units"]))
print(tx(TXT["sa"]).format(sa.iloc[-1]))
hist = np.asarray(RESULT["saar_hist"], dtype=float)
print(tx(TXT["saar"]).format(float(pd.Series(hist).dropna().iloc[-1])))
print(tx(TXT["med"]).format(float(np.nanmedian(hist[-36:]))))

print(tx(TXT["open"]))
# Сводка не должна падать, если разделы 7 и 8 не запускались в этой сессии: блокнот часто
# перезапускают с середины. | The summary must not fail if sections 7 and 8 were not run in
# this session: the notebook is often re-run from the middle.
if "cand" in dir():
    edge = cand[cand["edge"] == True]
    print(tx(TXT["edge"]).format(", ".join(edge["date"])) if len(edge) else tx(TXT["noedge"]))
else:
    print(tx(TXT["skip"]).format("7"))

if "tbl" in dir():
    row = tbl.loc[tbl["engine"] == S["decompose"]["engine"], "revision_pct"]
    if len(row) and np.isfinite(row.iloc[0]):
        print(tx(TXT["rev"]).format(float(row.iloc[0])))
else:
    print(tx(TXT["skip"]).format("8"))
